## Use requests to get data

In [ ]:
import requests # HTTP library, used to call the openFDA API

In [ ]:
# 调用 openFDA 药品标签接口，limit=5 表示只取 5 份标签
drug = requests.get("https://api.fda.gov/drug/label.json", params={"limit": 5})

drug.raise_for_status()  # 如果接口返回错误（如 404、429 限流），在这里直接报错，而不是后面出现难懂的 KeyError

In [18]:
data = drug.json()  # 把返回的 JSON 文本解析成 Python 字典

labels = data["results"]  # 标签列表在 "results" 键下，每个元素是一份标签（dict）

In [19]:
labels[0].keys()  # 查看第一份标签有哪些字段

dict_keys(['effective_time', 'inactive_ingredient', 'purpose', 'keep_out_of_reach_of_children', 'warnings', 'questions', 'spl_product_data_elements', 'openfda', 'version', 'dosage_and_administration', 'pregnancy_or_breast_feeding', 'stop_use', 'storage_and_handling', 'do_not_use', 'package_label_principal_display_panel', 'indications_and_usage', 'set_id', 'id', 'active_ingredient'])

In [20]:
labels[0]["indications_and_usage"]  # 查看适应症字段的内容（是一个字符串列表）

['INDICATIONS Condition listed above or as directed by the physician']

In [7]:
#labels[0]["openfda"]["generic_name"][0]

In [21]:
# how to deal with the case when the key doesn't exist in the dictionary? Use .get(key, default) to avoid KeyError.
# do not assume that the key exists, length of the list, or has fallback values.

def get_drug_name(label: dict) -> str:
    """
    Extracts the drug name from the label dictionary.
    
    Args:
        label (dict): A dictionary containing drug label information.

    Returns:
        str: The drug name.
    """
    # .get(key, default) is used to avoid KeyError if the key doesn't exist
    openfda = label.get("openfda", {})  # 取 openfda 子字典，没有就用空字典
    # 按优先级找名字：通用名 → 品牌名 → 成分名；`or` 会返回第一个非空的值，都为空时返回 []
    names = openfda.get("generic_name") or openfda.get("brand_name") or openfda.get("substance_name") or []
    return ", ".join(names) if names else "Unknown Drug Name"  # 多个名字用逗号连接；一个都没有就返回占位文字

In [9]:
# validate the function by printing the drug names for each label
for i, l in enumerate(labels):
    print(i, l.get("openfda", {}).get("generic_name", "【缺失】"))

0 ['SILICEA']
1 【缺失】
2 ['POVIDONE-IODINE']
3 【缺失】
4 【缺失】


In [10]:
for i, l in enumerate(labels):
    print(i, get_drug_name(l))

0 SILICEA
1 None
2 POVIDONE-IODINE
3 None
4 None


In [11]:
labels[1].get("openfda")

{}

## Define a Schema using Pydantic

In [22]:
from pydantic import BaseModel, Field  # BaseModel 定义数据结构；Field 给字段加默认值和描述
from typing import Literal, Optional   # Literal 限定只能取几个固定值；Optional[X] 表示 X 或 None

# 把白名单内的标签段落合并成一段文本（字段列表见 label_to_text.py 里的 FIELDS）
import importlib, label_to_text as _l2t  # importlib 用来重新加载模块
importlib.reload(_l2t)  # 修改 label_to_text.py 后不用重启 kernel 也能生效（Python 默认会缓存已导入的模块）
from label_to_text import label_to_text  # 从重新加载后的模块里导入函数

In [31]:
# Schema 的结构对应 FDA 标签里安全信息的分级，
# 让模型每类信息都有地方放，而不是全部塞进一个列表

# 允许的剂型：模型只能从这里选，避免出现 "Pellets" / "Bark" / "ophthalmic solution" 这种五花八门的写法
DosageForm = Literal[
    "tablet", "capsule", "pellet", "powder", "liquid", "solution", "suspension",  # 片剂、胶囊、小丸、粉剂、液体、溶液、混悬液
    "syrup", "cream", "ointment", "gel", "lotion", "spray", "drops",             # 糖浆、乳膏、软膏、凝胶、洗剂、喷雾、滴剂
    "injection", "patch", "suppository", "other",                                # 注射剂、贴剂、栓剂、其他
]

class SafetyInfo(BaseModel):  # 安全信息，分三级
    contraindications: list[str] = Field(  # 第一级：绝对禁忌
        default_factory=list,  # 默认是空列表（用 default_factory 避免多个实例共用同一个列表）
        description="Absolute prohibitions: conditions under which the drug must never be used "  # 这段描述会发给模型，告诉它该放什么
        "(from CONTRAINDICATIONS on Rx labels or DO_NOT_USE on OTC labels), e.g. known allergy "  # 相邻字符串会自动拼成一个
        "to an ingredient. Empty list if the label has none.",
    )
    cautions: list[str] = Field(  # 第二级：慎用，需要咨询医生
        default_factory=list,  # 默认空列表
        description="Conditions requiring care or a doctor's advice before use "  # 给模型的说明
        "(ASK_DOCTOR, WARNINGS), e.g. pregnancy or breast-feeding, liver disease.",
    )
    stop_use_conditions: list[str] = Field(  # 第三级：出现某情况时停药
        default_factory=list,  # 默认空列表
        description="Situations in which the user should stop taking the drug "  # 给模型的说明
        "(STOP_USE), e.g. symptoms persist more than N days.",
    )

class DrugLabel(BaseModel):  # 模型最终要返回的完整结构
    indications: list[str] = Field(  # 适应症
        default_factory=list,  # 默认空列表
        description=(  # 给模型一条明确的决策规则，消除“这句算不算适应症”的歧义
            "Specific medical conditions or symptoms this product treats or relieves. "  # 要什么：具体的病症或症状
            "Look in INDICATIONS_AND_USAGE and PURPOSE sections. "  # 去哪找：告诉模型看哪两个段落
            "Exclude non-informative phrases such as 'condition listed above' or "  # 不要什么：没有信息量的套话
            "'as directed by the physician'. "
            "Each item must be an independently meaningful condition "  # 每一项单独拿出来也要有意义
            "(e.g. 'acne', not 'temporary relief')."  # 正反例各一个，比抽象规则更好用
        ),
    )
    safety: SafetyInfo = Field(  # 嵌套上面定义的 SafetyInfo
        description="Safety statements about the patient. Never include packaging, tamper "  # 明确排除包装、封口、储存等非人体相关信息
        "seals, storage, 'external use only', or 'keep out of reach of children'.",
    )
    dosage_form: Optional[DosageForm] = Field(  # 剂型：只能是 DosageForm 里的值，或 None
        default=None,  # 标签没写时为 None
        description="Physical form of the product as administered, taken from dosage or "  # 让模型从用法里判断剂型
        "administration text (e.g. 'dissolve under the tongue' pellets => pellet). "
        "Not the plant part or raw ingredient. Null if not stated.",  # 防止把原料部位（如 BARK 树皮）当成剂型
    )

## OpenAI API

In [24]:
from openai import OpenAI  # OpenAI 官方 SDK，用来调用模型
from dotenv import load_dotenv  # 从 .env 文件读取环境变量（例如 OPENAI_API_KEY）

In [ ]:
load_dotenv()  # 把 .env 里的变量加载到环境变量中

client = OpenAI()  # 创建客户端；会自动从环境变量读取 OPENAI_API_KEY

# 系统提示词：告诉模型它的角色，以及只能使用原文里的信息（减少编造）
SYSTEM_PROMPT = "You extract structured information from FDA drug labels. Only use information present in the text."

# 没有信息量的套话：包含这些短语的 indication 会被代码直接删掉（全部用小写，方便比较）
NON_INFORMATIVE = {"condition listed above", "as directed by the physician", "as directed"}

def clean_indications(items: list[str]) -> list[str]:
    """后处理：删掉套话，并去掉重复项（忽略大小写和首尾空格）。"""
    cleaned = []  # 保存保留下来的条目
    seen = set()  # 记录已经出现过的条目（小写形式），用来去重
    for item in items:  # 逐条检查模型给出的 indication
        key = item.strip().lower()  # 统一成小写、去掉首尾空格，便于比较
        if not key:  # 空字符串
            continue  # 跳过
        if any(p in key for p in NON_INFORMATIVE):  # 只要包含任意一个套话短语
            continue  # 就丢掉这一条
        if key in seen:  # 之前已经出现过（例如 "Acne" 和 "acne"）
            continue  # 丢掉重复项
        seen.add(key)  # 记录下来
        cleaned.append(item.strip())  # 保留原始大小写，只去掉首尾空格
    return cleaned  # 返回清洗后的列表

def extract_drug_label_info(label_text: str, clean: bool = True) -> DrugLabel | None:
    """
    Extract structured information from a drug label text using OpenAI API.
    clean=True 时对 indications 做规则过滤；clean=False 返回模型原始结果（用来对比过滤的效果）。
    Returns None if the model refuses or its output can't be parsed into DrugLabel.
    """
    response = client.responses.parse(  # Responses API + 结构化输出：SDK 会把 DrugLabel 转成 JSON Schema 发给模型
        model="gpt-4o-mini",  # 使用的模型
        temperature=0,  # 降低随机性；但它解决不了“两种答案本来就一样合理”的歧义，歧义要在输入侧解决
        input=[  # 对话消息列表
            {"role": "system", "content": SYSTEM_PROMPT},  # 系统消息：规则
            {"role": "user", "content": label_text}  # 用户消息：标签文本
        ],
        text_format=DrugLabel,  # 要求模型按 DrugLabel 的结构输出
    )

    # Responses API 没有 .choices（那是 Chat Completions API 的写法）
    # output_parsed 已经是 DrugLabel 实例；output_text 是模型返回的原始 JSON 字符串
    result = response.output_parsed  # 取出解析好的结果
    if result is None:  # 模型拒绝回答或输出不符合 schema 时为 None
        print("Could not parse model output:", response.output_text)  # 打印原始输出，方便排查
        return None  # 直接返回 None
    if clean:  # 需要清洗时
        result.indications = clean_indications(result.indications)  # 用规则过滤 indications
    return result  # 返回 DrugLabel 实例

In [ ]:
for i, label in enumerate(labels):  # enumerate 同时给出序号 i 和标签 label
    label_text = label_to_text(label)  # 把白名单字段合并成一段文本
    print(f"===== label {i}: {get_drug_name(label)} ({len(label_text)} chars) =====")  # 打印序号、药名和 prompt 长度（字符数，可粗略反映成本）
    result = extract_drug_label_info(label_text)  # 调用模型抽取结构化信息
    if result is None:  # 解析失败
        continue  # 跳过这份标签，继续下一份
    print(result.model_dump_json(indent=2))  # 把结果转成带缩进的 JSON 打印
    print("original label text:", label_text, "\n")  # 打印原文，方便人工对照检查

===== label 0: SILICEA (1055 chars) =====
{
  "indications": [
    "Condition listed above or as directed by the physician"
  ],
  "safety": {
    "contraindications": [
      "Do not use if capseal is broken or missing"
    ],
    "cautions": [
      "If you are pregnant, or nursing a baby, seek professional advice before taking this product"
    ],
    "stop_use_conditions": [
      "If symptoms do not improve in 4 days, or worsen, discontinue use and seek assistance of health professional"
    ]
  },
  "dosage_form": "pellet"
}
original label text: INDICATIONS_AND_USAGE:
INDICATIONS Condition listed above or as directed by the physician

PURPOSE:
USES USES: Temporary Relief - Acne, Boils* * Claims based on traditional homeopathic practice, not accepted medical evidence. Not FDA evaluated.

DO_NOT_USE:
Do not use if capseal is broken or missing. Close the cap tightly after use.

WARNINGS:
WARNINGS This product is to be used for self-limiting conditions If symptoms do not improve in 4

In [ ]:
# for run in range(3):
#     result = extract_drug_label_info(label_to_text(labels[0]))
#     print(f"--- run {run} ---")
#     print(result.model_dump_json(indent=2))

--- run 0 ---
{
  "indications": [
    "Condition listed above or as directed by the physician"
  ],
  "safety": {
    "contraindications": [
      "Do not use if capseal is broken or missing."
    ],
    "cautions": [
      "If you are pregnant, or nursing a baby, seek professional advice before taking this product."
    ],
    "stop_use_conditions": [
      "If symptoms do not improve in 4 days, or worsen, discontinue use and seek assistance of health professional."
    ]
  },
  "dosage_form": "pellet"
}
--- run 1 ---
{
  "indications": [
    "Temporary Relief - Acne",
    "Boils"
  ],
  "safety": {
    "contraindications": [
      "Do not use if capseal is broken or missing."
    ],
    "cautions": [
      "If you are pregnant, or nursing a baby, seek professional advice before taking this product."
    ],
    "stop_use_conditions": [
      "If symptoms do not improve in 4 days, or worsen, discontinue use and seek assistance of health professional."
    ]
  },
  "dosage_form": "pell

In [ ]:
# 一致性测试：每份标签跑 3 次，检查 indications 是否稳定
# 同时记录“过滤前”（模型原始输出）和“过滤后”，区分稳定性来自 description 规则还是代码过滤
N_RUNS = 3  # 每份标签跑几次

def as_set(items: list[str]) -> frozenset:
    """把列表转成忽略大小写、忽略顺序的集合，只比较内容。"""
    return frozenset(i.strip().lower() for i in items)  # frozenset 可以放进 set 里做比较

for i, label in enumerate(labels):  # 遍历 5 份标签
    label_text = label_to_text(label)  # 生成 prompt 文本
    raw_runs, clean_runs = [], []  # 分别保存每次的原始结果和过滤后结果
    for run in range(N_RUNS):  # 跑 N_RUNS 次
        result = extract_drug_label_info(label_text, clean=False)  # 取原始结果（不过滤）
        if result is None:  # 解析失败
            continue  # 这次不计入
        raw_runs.append(result.indications)  # 保存原始 indications
        clean_runs.append(clean_indications(result.indications))  # 用同一次结果做过滤，保证对比公平

    raw_stable = len({as_set(r) for r in raw_runs}) == 1  # 所有次的集合都相同 → 稳定
    clean_stable = len({as_set(r) for r in clean_runs}) == 1  # 过滤后是否稳定
    print(f"===== label {i}: {get_drug_name(label)} =====")  # 标题
    print(f"raw stable: {raw_stable} | cleaned stable: {clean_stable}")  # 一眼看出结论
    for run, (raw, cleaned) in enumerate(zip(raw_runs, clean_runs), start=1):  # 逐次打印细节
        print(f"  run {run} raw    : {raw}")  # 模型原始输出
        print(f"  run {run} cleaned: {cleaned}")  # 过滤后的结果
    print()  # 空一行，分隔不同标签